# Imports

In [1]:
import os
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Loader

In [5]:
CLASSES = ["Drinking", "Walking", "Cooking", "Exercise", "Ironing", "Phone-Conversation", "Reading-Book", "Using-Computer", "Vacuum-Cleaning", "Watching-TV"]

root_dir = "../custom_IMSHA_Dataset/"
samples = []
for dir_name in os.listdir(root_dir):
    label = dir_name.split('_')[0]
    full_path = os.path.join(root_dir, dir_name)
    if os.path.isdir(full_path):
        samples.append((full_path, label))

imu_data = pd.DataFrame(columns=['ax1','ay1','az1','gx1','gy1','gz1','ax2','ay2','az2','gx2','gy2','gz2','label'])
imu_data.dropna(inplace=True)
for s in samples:
    path, label_str = s
    imu = pd.read_csv(os.path.join(path, 'out.csv'))
    imu = imu.loc[:, ~imu.columns.str.contains('^Unnamed')]
    imu = imu.iloc[:, :12]  
    imu['label'] = CLASSES.index(label_str)
    imu_data = imu_data._append(imu, ignore_index=True)
imu_data.head()

/var/folders/0y/4sn48ws54fvc3hldm5bsrgj00000gn/T/ipykernel_63314/982302731.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  imu_data = imu_data._append(imu, ignore_index=True)


,ax1,ay1,az1,gx1,gy1,gz1,ax2,ay2,az2,gx2,gy2,gz2,label
0,0.52244,9.54094,1.40611,0.04274,-0.02754,0.01496,1.24498,9.71088,0.74699,-0.08983,-0.29677,0.22234,2
1,0.35715,9.55292,1.13302,0.05872,-0.05097,0.01070,1.55144,9.69173,0.45969,-0.03123,-0.10075,-0.02693,2
2,0.41704,9.68227,1.11386,0.10558,-0.08612,0.00324,1.47482,9.48103,0.28731,0.00179,0.08673,-0.07169,2
3,0.27810,9.57687,1.11146,0.14285,-0.15108,0.02454,1.32159,9.53850,0.74699,-0.08025,0.16237,-0.10045,2
4,0.39788,9.77570,0.95096,0.19397,-0.18942,0.03093,0.95769,9.63426,0.28731,-0.02165,0.34453,-0.13666,2


# Split


In [3]:
import numpy as np

imu_data['label'] = imu_data['label'].astype('category')
X = imu_data.iloc[:, :-1]
y = imu_data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Unique values of y:", np.unique(y_train))

Unique values of y: [0 1 2 3 4 5 6 7 8 9]


# Train


In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
param_grid = {
    'learning_rate': [0.1, 0.01],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}
number_of_cross_validation_folds=5
xgb_classifier = xgb.XGBClassifier(objective='multi:softmax',  num_class=len(CLASSES))
xgb_classifier.fit(X_train, y_train)
predictions = xgb_classifier.predict(X_test)
print(classification_report(y_test,predictions))

accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

grid_search = GridSearchCV(xgb_classifier, param_grid=param_grid, cv=number_of_cross_validation_folds)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

              precision    recall  f1-score   support

           0       0.77      0.77      0.77      3648
           1       0.78      0.76      0.77      3816
           2       0.69      0.75      0.72      3736
           3       0.75      0.74      0.75      3807
           4       0.77      0.70      0.73      3790
           5       0.79      0.81      0.80      3859
           6       0.87      0.83      0.85      3869
           7       0.78      0.84      0.81      3625
           8       0.84      0.85      0.85      3861
           9       0.95      0.96      0.96      3776

    accuracy                           0.80     37787
   macro avg       0.80      0.80      0.80     37787
weighted avg       0.80      0.80      0.80     37787

Accuracy: 0.8006986529758912


KeyboardInterrupt: 

# Eval

In [ ]:
input_df = pd.read_csv('../custom_IMSHA_Dataset/Walking/out.csv')
input_df = input_df[['ax1', 'ay1', 'az1', 'gx1', 'gy1', 'gz1',
                   'ax2', 'ay2', 'az2', 'gx2', 'gy2', 'gz2']]
output_data = pd.DataFrame([], columns=['Activity'])
imu_dataset = input_df.iloc[:,:12]
for i in imu_dataset.index.values.tolist():
    output = xgb_classifier.predict_proba([imu_dataset.iloc[i]])
    predictions = output.argmax(axis=1)
    confidence_scores = output.max(axis=1)
    predicted_classes = [xgb_classifier.classes_[pred] for pred in predictions][0]
    # Set the confidence threshold
    confidence_threshold = 0.80
    
    predicted_class = "Idle"
    if confidence_scores >=confidence_threshold:
        predicted_class = CLASSES[predicted_classes]
    
    output_data.loc[len(output_data.index)] = [predicted_class]
    #print([imu_dataset.iloc[i]])
    print('Activity = {}, Confidence: {}'.format(predicted_class, confidence_scores))
#output_data.to_csv('../collected/xg_final_output.csv')

# Save

# Load